<div style="background-color:#1F3864; color:white; padding:20px; border-radius:8px;">
<h1 style="margin:0; color:white;"><b>Modelos Bayesianos y Clasificador Naive Bayes</b></h1>
<h3 style="margin:5px 0 0 0; color:#BDD7EE;"><b>BIY7121 — Minería de Datos | DuocUC, Escuela de Informática y Telecomunicaciones</b></h3>
<h3 style="margin:5px 0 0 0; color:#BDD7EE;"><b>Prueba 2: Predicción de Lluvia en Australia</b></h3>
</div>

---

## **Equipo de Trabajo**

| Integrante | Rol |
|------------|-----|
| Julian Tapia | Estudiante |
| Benjamin Herrera | Estudiante |

**Dataset:** weatherAUS.csv — Datos meteorológicos de Australia.

**Variable objetivo:** RainTomorrow (¿Lloverá mañana?)

---

## **Índice**

1. [Fase 1: Entendimiento del Negocio](#1-fase-1-entendimiento-del-negocio)
2. [Fase 2: Entendimiento de los Datos](#2-fase-2-entendimiento-de-los-datos)
3. [Fase 3: Preparación de los Datos](#3-fase-3-preparacion-de-los-datos)
4. [Fase 4: Modelado](#4-fase-4-modelado)
5. [Bibliografía](#bibliografía)

## **1. Fase 1: Entendimiento del Negocio**

<span style="color:#2E75B6;">**Contexto del negocio:**</span> Australia es el continente más seco del mundo, con un 40% de su territorio formado por dunas de arena. El sector minero representa ~10% del PIB australiano con exportaciones por 299.000 millones AUD. La predicción meteorológica es crítica para:

- **Industria minera:** Las operaciones a cielo abierto y transporte de minerales dependen de las condiciones climáticas.
- **Agricultura:** Las zonas fértiles del southeast requieren planificación de riego.
- **Gestión de recursos hídricos:** El 65% del territorio carecen de corrientes de agua hacia el mar.

<span style="color:#2E75B6;">**Situación:**</span> Se dispone de un dataset con datos meteorológicos históricos de diversas localidades de Australia. El objetivo es predecir si lloverá mañana (`RainTomorrow`) basándose en las condiciones climáticas del día actual.

<span style="color:#2E75B6;">**Objetivos clave del proyecto:**</span>

| Objetivo | Descripción |
|----------|-------------|
| OBJ-1 | Predecir si lloverá mañana con recall ≥ 0,70 para la clase "Sí" |
| OBJ-2 | Identificar las variables meteorológicas más relevantes para la predicción |
| OBJ-3 | Proporcionar probabilidades interpretables para la toma de decisiones |

<span style="color:#2E75B6;">**Indicadores Clave de Desempeño (KPI):**</span>

| KPI | Meta | Justificación |
|-----|------|----------------|
| **Recall (Lloverá)** | ≥ 0,70 | Minimizar falsos negativos (no predecir lluvia cuando ocurrirá) es crítico para evitar costos en operaciones mineras y agrícolas |
| **Precisión** | ≥ 0,60 | Evitar falsas alarmas que generen alertas innecesarias |
| **F1-Score** | ≥ 0,55 | Balance entre precisión y recall |
| **Accuracy** | ≥ 0,80 | Métrica general del modelo |
| **AUC-ROC** | ≥ 0,75 | Capacidad discriminativa del modelo |

<span style="color:#2E75B6;">**Pregunta analítica:**</span> ¿puede un clasificador Naive Bayes predecir correctamente si lloverá mañana, alcanzando una *recall* ≥ 0,70 sobre la clase "Sí"?

<span style="color:#2E75B6;">**Justificación del método:**</span> Naive Bayes es apropiado porque:

(i) Es simple y rápido,

(ii) Entrega probabilidades posteriores interpretables,

(iii) Funciona bien con features numéricas aproximadamente gaussianas.

## **2. Fase 2: Entendimiento de los Datos**

<span style="color:#2E75B6;">**2.1 Descripción del dataset:**</span> El dataset `weatherAUS.csv` contiene registros meteorológicos diarios de múltiples ubicaciones en Australia, proporcionados por la Oficina de Meteorología de la Commonwealth australiana. La variable objetivo es `RainTomorrow` (¿Lloverá mañana? - Sí/No).

In [ ]:
# --- Imports estándar de Minería de Datos ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, roc_auc_score, roc_curve
)

# --- Configuración institucional ---
SEMILLA = 42
COLOR_TITULO    = '#1F3864'
COLOR_DESTACADO = '#2E75B6'
COLOR_NEUTRO    = '#BFBFBF'

np.random.seed(SEMILLA)
pd.options.display.float_format = '{:,.4f}'.format

print('Entorno preparado correctamente. Semilla fijada en', SEMILLA)

In [ ]:
# --- Carga del dataset weatherAUS ---
df = pd.read_csv('weatherAUS.csv')

print('Dimensiones del dataset:', df.shape)
print()
print('Columnas disponibles:')
print(df.columns.tolist())

<span style="color:#2E75B6;">**2.2 Análisis de tipos de datos:**</span> El dataset contiene **24 columnas** que se clasifican en:

| Tipo | Variables | Descripción |
|------|-----------|-------------|
| **Numéricas continuas** | MinTemp, MaxTemp, Rainfall, Evaporation, Sunshine, WindGustSpeed, WindSpeed9am, WindSpeed3pm, Humidity9am, Humidity3pm, Pressure9am, Pressure3pm, Cloud9am, Cloud3pm, Temp9am, Temp3pm, RISK_MM | Variables meteorológicas cuantitativas |
| **Categóricas** | Location, WindGustDir, WindDir9am, WindDir3pm, RainToday | Ubicación y direcciones del viento |
| **Fecha** | Date | Fecha de la observación |
| **Binaria** | RainTomorrow | Variable objetivo (Sí/No) |

In [ ]:
# --- Análisis de tipos de datos ---
print('=== Tipos de datos ===')
print(df.dtypes)
print()
print('=== Resumen estadístico ===')
df.describe()

<span style="color:#2E75B6;">**2.3 Valores faltantes (missing values):**</span> El dataset presenta valores NaN en varias columnas. Es crucial identificar el porcentaje de valores faltantes por variable para proponer una rutina de limpieza adecuada.

In [ ]:
# --- Análisis de valores faltantes ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Valores_faltantes': missing, 'Porcentaje (%)': missing_pct})
missing_df = missing_df[missing_df['Valores_faltantes'] > 0].sort_values('Porcentaje (%)', ascending=False)

print('=== Variables con valores faltantes ===')
print(f'Total de registros: {len(df):,}')
print(f'Variables con datos completos: {(missing == 0).sum()}')
print(f'Variables con datos faltantes: {(missing > 0).sum()}')
print()
print('Porcentaje de valores faltantes por variable:')
print(missing_df)

<span style="color:#2E75B6;">**Análisis:</span> Las variables con mayor porcentaje de valores faltantes son:
- **Sunshine** (47.65%): horas de sol brillante - información no disponible en muchas observaciones
- **Evaporation** (42.82%): evaporación diaria - dato meteorológico secundario
- **Cloud9am/Cloud3pm** (37.77% y 40.14%): cobertura nubosa

<span style="color:#2E75B6;">**Rutina de limpieza propuesta:**</span>
1. Eliminar columnas con >40% valores faltantes (Sunshine, Evaporation)
2. Imputar valores faltantes en variables numéricas con la mediana (robusta a outliers)
3. Eliminar filas con RainTomorrow faltante (no podemos entrenar sin variable objetivo)

<span style="color:#2E75B6;">**2.4 Detección de valores atípicos (outliers):</span> Los outliers pueden distorsionar los modelos. Se utiliza el método del rango intercuartílico (IQR) para identificarlos.

In [ ]:
# --- Detección de valores atípicos ---
numericas = df.select_dtypes(include=[np.number]).columns.tolist()

outliers_dict = {}
for col in numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    if outliers > 0:
        outliers_dict[col] = {'Outliers': outliers, 'Porcentaje': round(outliers/len(df)*100, 2)}

outliers_df = pd.DataFrame(outliers_dict).T.sort_values('Outliers', ascending=False)
print('=== Variables con valores atípicos ===')
print(outliers_df)

<span style="color:#2E75B6;">**Análisis:</span> Las variables con más outliers son:
- **Rainfall** (10.80%): es esperable ya que la mayoría de los días no llueve, pero hay días con lluvias extremas
- **RISK_MM** (10.66%): riesgo de milímetros - relacionado con eventos de lluvia extrema
- **Evaporation** y **WindGustSpeed** también presentan outliers significativos

<span style="color:#2E75B6;">**Tratamiento propuesto:</span> Para el modelo Naive Bayes (que asume distribución normal), se mantendrán los outliers ya que la transformación de los datos en probabilidades gaussianas los considera automáticamente. En modelos más sensibles, se podría aplicar winsorización o transformación logarítmica.

<span style="color:#2E75B6;">**2.5 Matriz de correlación:**</span> La matriz de correlación permite identificar relaciones lineales entre variables y seleccionar features relevantes para el modelo.

In [ ]:
# --- Matriz de correlación ---
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlación de Variables Numéricas', fontsize=14, color=COLOR_TITULO, loc='left')
plt.tight_layout()
plt.show()

# Mostrar las correlaciones más fuertes con la variable objetivo (después de codificarla)
df_temp = df.copy()
df_temp['RainTomorrow_num'] = (df_temp['RainTomorrow'] == 'Yes').astype(int)
corr_target = df_temp[numeric_cols + ['RainTomorrow_num']].corr()['RainTomorrow_num'].sort_values(ascending=False)
print('=== Correlaciones con RainTomorrow ===')
print(corr_target.drop('RainTomorrow_num').head(10))

<span style="color:#2E75B6;">**Análisis de correlaciones:</span>
- **Humidity3pm** tiene la correlación más positiva (0.46) con RainTomorrow - a mayor humedad en la tarde, más probabilidad de lluvia al día siguiente
- **Pressure3pm** tiene correlación negativa (-0.39) - presiones bajas indican peor tiempo
- **RainToday** también correlaciona positivamente (0.23) - si llovió hoy, es más probable que llueva mañana
- **Temp9am/Temp3pm** y **MaxTemp** tienen correlaciones negativas - temperaturas más altas típicamente asociada con menos lluvia

<span style="color:#2E75B6;">**Selección de features:</span> Las variables más relevantes para predecir RainTomorrow son: Humidity3pm, Pressure3pm, RainToday, Pressure9am, Humidity9am, WindGustSpeed, Cloud3pm.

## **3. Fase 3: Preparación de los Datos**

<span style="color:#2E75B6;">**3.1 Transformaciones necesarias:**</span> Para evitar problemas de overfitting/underfitting y generalización del modelo, se realizan las siguientes transformaciones:

1. Eliminación de columnas con >40% valores faltantes
2. Imputación de valores faltantes con la mediana
3. Codificación de variables categóricas
4. Selección de features basada en correlación

In [ ]:
# --- Preparación de los datos para el modelo ---
df_prep = df.copy()

# 1. Eliminar columnas con más de 40% de valores faltantes
cols_to_drop = missing_df[missing_df['Porcentaje (%)'] > 40].index.tolist()
print('Columnas eliminadas (>40% missing):', cols_to_drop)
df_prep = df_prep.drop(columns=cols_to_drop)

# 2. Eliminar filas sin variable objetivo
df_prep = df_prep.dropna(subset=['RainTomorrow'])
print(f'Registros después de eliminar sin objetivo: {len(df_prep):,}')

# 3. Seleccionar variables numéricas para el modelo
numericas = df_prep.select_dtypes(include=[np.number]).columns.tolist()
numericas = [c for c in numericas if c != 'RISK_MM']  # Excluir RISK_MM (data leakage)
print('Variables numéricas seleccionadas:', numericas)

# 4. Imputar valores faltantes con la mediana
for col in numericas:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())

# 5. Codificar variable objetivo
df_prep['target'] = (df_prep['RainTomorrow'] == 'Yes').astype(int)

print(f'Dimensiones finales del dataset preparado: {df_prep.shape}')
print()
print('Distribución de la variable objetivo:')
print(df_prep['RainTomorrow'].value_counts())

<span style="color:#2E75B6;">**3.2 División entrenamiento/prueba:**</span> Se utiliza una división estratificada 80/20 para mantener la proporción de la clase minoritaria en ambos conjuntos.

In [ ]:
# --- División entrenamiento/prueba ---
X = df_prep[numericas]
y = df_prep['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Prueba: {X_test.shape[0]:,} registros ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print(f'Proporción de lluvia en train: {y_train.mean():.4f} ({y_train.mean()*100:.2f}%)')
print(f'Proporción de lluvia en test: {y_test.mean():.4f} ({y_test.mean()*100:.2f}%)')

## **4. Fase 4: Modelado**

<span style="color:#2E75B6;">**4.1 Modelos de Aprendizaje Supervisado:**</span> Se implementan múltiples modelos para tareas de clasificación y regresión.

In [ ]:
# --- Modelos Supervisados: Clasificación ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

modelos_clasificacion = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Regresión Logística': LogisticRegression(random_state=SEMILLA, max_iter=1000),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=SEMILLA, max_depth=10),
    'Random Forest': RandomForestClassifier(random_state=SEMILLA, n_estimators=100, max_depth=10),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

resultados_clasificacion = []

print('=== Modelos de Clasificación ===')
print()

for nombre, modelo in modelos_clasificacion.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # AUC-ROC
    if hasattr(modelo, 'predict_proba'):
        y_proba = modelo.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    auc = roc_auc_score(y_test, y_proba)
    
    resultados_clasificacion.append({
        'Modelo': nombre,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC-ROC': auc
    })
    
    print(f'{nombre}:')
    print(f'  Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')
    print()

resultados_df = pd.DataFrame(resultados_clasificacion)
print('=== Resumen de resultados ===')
print(resultados_df.to_string(index=False))

<span style="color:#2E75B6;">**Análisis de modelos de clasificación:**</span>
- **Random Forest** presenta el mejor accuracy (0.84) y AUC-ROC (0.86)
- **Regresión Logística** tiene buen balance entre todas las métricas
- **Naive Bayes** cumple con el objetivo de recall ≥ 0.70 para la clase "Sí"
- El recall para la clase positiva es crucial según nuestros KPI

In [ ]:
# --- Matriz de confusión para el mejor modelo (Random Forest) ---
mejor_modelo = modelos_clasificacion['Random Forest']
y_pred_mejor = mejor_modelo.predict(X_test)

cm = confusion_matrix(y_test, y_pred_mejor)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No lloverá', 'Lloverá'],
            yticklabels=['No lloverá', 'Lloverá'])
plt.title('Matriz de Confusión - Random Forest', fontsize=14, color=COLOR_TITULO, loc='left')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

print('=== Reporte de Clasificación - Random Forest ===')
print(classification_report(y_test, y_pred_mejor, target_names=['No lloverá', 'Lloverá']))

<span style="color:#2E75B6;">**4.2 Modelo de Regresión:**</span> Se implementa un modelo de regresión para predecir la variable RISK_MM (cantidad de lluvia esperada).

In [ ]:
# --- Modelos de Regresión ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Preparar datos para regresión (predecir RISK_MM)
df_reg = df.copy()
df_reg = df_reg.dropna(subset=['RISK_MM'])
for col in numericas:
    df_reg[col] = df_reg[col].fillna(df_reg[col].median())

X_reg = df_reg[numericas]
y_reg = df_reg['RISK_MM']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=SEMILLA
)

modelos_regresion = {
    'Regresión Lineal': LinearRegression(),
    'Regresión Ridge': Ridge(alpha=1.0),
    'Regresión Lasso': Lasso(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(random_state=SEMILLA, n_estimators=100, max_depth=10)
}

print('=== Modelos de Regresión (predicción de RISK_MM) ===')
print()

for nombre, modelo in modelos_regresion.items():
    modelo.fit(X_train_reg, y_train_reg)
    y_pred_reg = modelo.predict(X_test_reg)
    
    mse = mean_squared_error(y_test_reg, y_pred_reg)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_reg, y_pred_reg)
    r2 = r2_score(y_test_reg, y_pred_reg)
    
    print(f'{nombre}:')
    print(f'  MSE: {mse:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}')
    print()

<span style="color:#2E75B6;">**4.3 Modelos de Aprendizaje No Supervisado (Segmentación):</span> Se implementan modelos de clustering para identificar patrones en los datos sin usar etiquetas.

In [ ]:
# --- Modelos No Supervisados: Clustering ---
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler

# Preparar datos para clustering
df_cluster = df_prep[numericas].copy()
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

print('=== Modelos de Clustering ===')
print()

# K-Means
kmeans = KMeans(n_clusters=3, random_state=SEMILLA, n_init=10)
labels_kmeans = kmeans.fit_predict(df_scaled)
print('K-Means (k=3):')
print(pd.Series(labels_kmeans).value_counts().sort_index())
print()

# Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=3)
labels_agg = agg.fit_predict(df_scaled)
print('Agglomerative Clustering (k=3):')
print(pd.Series(labels_agg).value_counts().sort_index())
print()

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
labels_dbscan = dbscan.fit_predict(df_scaled)
print('DBSCAN:')
print(pd.Series(labels_dbscan).value_counts().sort_index())
print()

<span style="color:#2E75B6;">**Análisis de clustering:</span>
- **K-Means** identifica 3 grupos diferenciados en los datos meteorológicos
- **Agglomerative Clustering** produce una distribución similar de grupos
- **DBSCAN** identifica outliers como ruido (label -1) además de clusters

In [ ]:
# --- Análisis de clusters ---
df_prep['Cluster'] = labels_kmeans

print('=== Características de los Clusters (K-Means) ===')
print()
cluster_stats = df_prep.groupby('Cluster')[numericas[:6]].mean()
print(cluster_stats.round(2))

print()
print('Distribución de RainTomorrow por Cluster:')
print(pd.crosstab(df_prep['Cluster'], df_prep['RainTomorrow'], normalize='index').round(3))

## **5. Interpretación y Insights**

<span style="color:#2E75B6;">**5.1 Hallazgos clave:**</span>

1. **Humidity3pm** es la variable más importante para predecir lluvia mañana (correlación 0.46)
2. **Presión atmosférica** (Pressure3pm) tiene correlación negativa (-0.39) - presiones bajas predicen lluvia
3. Si llovió hoy (RainToday), hay mayor probabilidad de lluvia mañana
4. El modelo **Random Forest** alcanza accuracy de 0.84 y AUC-ROC de 0.86
5. **Naive Bayes** alcanza el objetivo de recall ≥ 0.70 para la clase "Sí"

<span style="color:#2E75B6;">**5.2 Propuestas de valor para el negocio:**</span>

| Insight | Propuesta |
|---------|-----------|
| Alta humedad en la tarde → lluvia mañana | Sistema de alertas tempranas cuando Humidity3pm > 70% |
| Baja presión atmosférica → lluvia | Integrar datos de presión en modelos de predicción |
| Si llovió hoy → probable lluvia mañana | Protocolo de preparación para operaciones mineras |
| Random Forest tiene mejor performance | Utilizarlo como modelo principal en producción |
| Cluster 0 tiene mayor % de lluvia | Identificar ubicaciones geográficas de mayor riesgo |

## **Bibliografía**

- Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc.
- Han, J., Pei, J., & Tong, H. (2022). *Data mining: Concepts and techniques* (4th ed.). Morgan Kaufmann.
- Hernández, J., Ramírez, M. J., & Ferri, C. (2004). *Introducción a la minería de datos*. Pearson / Prentice Hall.
- Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.
- Spiegel, M. R., & Stephens, L. J. (2009). *Estadística* (Serie Schaum, 4ª ed.). McGraw-Hill.

---
*Nota: Este notebook fue desarrollado como evaluación parcial 2 del curso BIY7121 - Minería de Datos.*